<a href="https://colab.research.google.com/github/jud3-hub/first-rental-guide/blob/main/WORK_18_%EC%9D%B8%EC%88%98_L0_%EA%B8%B0%EB%B3%B8%EB%AA%A8%EB%8D%B8_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 전월세 챗봇 · 질문 3개 첫 실험

목표: Q01 · Q04 · Q05를 독립적으로 실행 → Google Drive에 저장 → 스프레드시트에서 수동 채점.

**시작 전:** Colab의 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후 아래 셀을 위에서부터 실행.

- 모델: `Qwen/Qwen2.5-1.5B-Instruct`. 양자화 · RAG · 파인튜닝 없이 시작.
- 법령과 기준 답변은 모델 입력에 포함하지 않음. 법령 JSON 업로드 불필요.
- 공개 모델 다운로드 방식으로 유료 LLM API 키 불필요. Colab 자원 이용 조건은 별도.
- 실행 결과는 자동 채점하지 않음. 채점표의 기준과 직접 대조.
- 이 노트북은 코드 구문 · 저장 로직 검증 완료. 실제 Colab GPU 추론은 미실행.

공식 모델 설명: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct


## 1. 라이브러리 설치

새 Colab 런타임에서 실행. 설치 후 재시작 안내가 나오면 런타임을 재시작한 뒤 2번부터 실행.


In [2]:
# %pip는 Colab 환경에서 외부 파이썬 패키지를 설치하는 매직 명령어입니다.
# -q 옵션은 설치 과정을 조용하게(quiet) 진행하여 로그 출력을 최소화합니다.
# 'transformers'는 Hugging Face의 LLM 모델을 다루기 위한 라이브러리이고,
# 'accelerate'는 PyTorch 모델의 하드웨어(GPU) 가속 및 메모리 효율적 로드를 돕는 라이브러리입니다.
# 여기서는 특정 버전(4.51.3과 1.6.0)을 지정하여 설치합니다.
%pip -q install "transformers==4.51.3" "accelerate==1.6.0"

## 2. Google Drive 연결

본인 계정의 Drive 연결 승인 필요. 매 실행마다 새로운 폴더를 만들어 이전 결과 보존.

공유 프로젝트의 `result/인수` 폴더를 먼저 확인하고 그 안에 결과 저장. 폴더가 보이지 않으면 경로를 확인한 뒤 실행.


In [4]:
# 필요한 라이브러리들을 불러옵니다.
from google.colab import drive   # Colab에 구글 드라이브를 연결하기 위한 모듈
from pathlib import Path         # 파일 및 디렉토리 경로를 객체로 다루기 위한 모듈
from datetime import datetime    # 날짜와 시간을 다루기 위한 모듈
from zoneinfo import ZoneInfo    # 특정 시간대(타임존) 설정을 위한 모듈
import json                      # 데이터를 JSON 형식으로 저장하고 읽기 위한 모듈
import time                      # 실행 시간 측정을 위한 모듈
import uuid                      # 중복되지 않는 고유한 ID(식별자)를 생성하기 위한 모듈
import unicodedata               # 한글 자모음 분리 현상(NFC/NFD)을 해결하기 위한 텍스트 정규화 모듈

# 1. 구글 드라이브를 '/content/drive' 경로에 마운트(연결)합니다.
# 실행 시 구글 계정 로그인 및 권한 승인 창이 뜹니다.
drive.mount('/content/drive')

# 2. 현재 한국 시간(Asia/Seoul)을 기준으로 '년월일_시분초' 형태의 문자열을 만들고,
# 중복 방지를 위해 무작위 UUID의 앞 6자리를 붙여 고유한 실험 ID(RUN_ID)를 생성합니다.
RUN_ID = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]

# 3. 결과를 저장할 구글 드라이브의 시작 경로를 지정합니다.
RESULT_ROOT = Path('/content/drive/MyDrive')

# 4. 지정된 폴더 트리('[01.공부]' -> '[AI] KT Cloud TECH UP' -> '[프로젝트 01]' -> 'result' -> '인수')를 차례대로 탐색합니다.
# 맥(Mac) 환경 등에서 발생할 수 있는 한글 자모음 분리 현상을 방지하기 위해 NFC 정규화를 거쳐 폴더명을 정확히 비교합니다.
for folder_name in ['[01.공부]', '[AI] KT Cloud TECH UP', '[프로젝트 01]', '05_result', '인수']:
    # 현재 위치(RESULT_ROOT) 하위 폴더 중 이름이 일치하는 폴더 목록을 찾습니다.
    matches = [p for p in RESULT_ROOT.iterdir()
               if p.is_dir() and unicodedata.normalize('NFC', p.name) == folder_name]

    # 폴더가 존재하지 않거나 여러 개 존재하면 에러를 발생시킵니다.
    if len(matches) != 1:
        raise FileNotFoundError(f'{RESULT_ROOT} 안의 {folder_name} 폴더를 확인해 주세요.')

    # 정상적으로 찾은 경우, 다음 탐색을 위해 탐색 루트를 해당 폴더 안으로 이동시킵니다.
    RESULT_ROOT = matches[0]

# 5. 최종 목적지 폴더 내부에 이번 실험 ID(RUN_ID) 이름으로 새로운 폴더 경로를 생성합니다.
SAVE_DIR = RESULT_ROOT / RUN_ID

# 6. 폴더가 실제로 드라이브에 생성됩니다. (이미 존재하면 에러 발생 옵션 사용)
SAVE_DIR.mkdir(parents=True, exist_ok=False)

# 7. 실험 정보와 저장 경로를 화면에 출력합니다.
print('실험 ID:', RUN_ID)
print('저장 위치:', SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
실험 ID: 20261030_113056_b4e767
저장 위치: /content/drive/MyDrive/[01.공부]/[AI] KT Cloud TECH UP/[프로젝트 01]/05_result/인수/20261030_113056_b4e767


## 3. 모델 1개 로딩

처음 실행 시 모델 다운로드에 시간이 걸림. GPU가 없으면 T4 GPU로 변경 후 다시 실행.


In [5]:
# PyTorch, Transformers 등 딥러닝 및 모델 로딩을 위한 라이브러리를 불러옵니다.
import torch, transformers, accelerate, platform
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM

# 현재 Colab 세션에 GPU(CUDA)를 사용할 수 있는지 확인합니다.
# GPU를 활성화하지 않았다면 경고 메시지와 함께 실행이 중단됩니다.
assert torch.cuda.is_available(), '런타임 유형을 T4 GPU로 변경해 주세요.'

# 실험에 사용할 모델의 Hugging Face ID를 정의합니다.
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
# 모델에게 줄 기본 역할(시스템 프롬프트)을 정의합니다.
SYSTEM_PROMPT = '사용자의 질문에 한국어로 답변하세요.'
# 현재 프롬프트의 버전을 명시합니다. (실험 비교용)
PROMPT_VERSION = 'baseline-v1'
# 모델이 한 번에 생성할 수 있는 최대 단어(토큰)의 수입니다.
MAX_NEW_TOKENS = 512

# 모델의 설정(Configuration)을 먼저 받아와 고유 버전 해시(commit hash)를 확인합니다.
config = AutoConfig.from_pretrained(MODEL_ID)
REVISION = config._commit_hash
assert REVISION, '모델 버전 확인 실패. 모델 로딩 셀을 다시 실행해 주세요.'

# 텍스트를 모델이 이해할 수 있는 숫자(토큰)로 변환해주는 토크나이저를 로드합니다.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)

# 1.5B 크기의 인공지능 모델을 불러옵니다.
# float16(반정밀도) 타입으로 불러와 메모리 사용량을 줄이고, GPU('cuda')에 올린 뒤, 평가 모드(.eval())로 전환합니다.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, torch_dtype=torch.float16
).to('cuda').eval()

# 나중에 실험을 메타데이터와 함께 기록하기 위해 주요 설정값을 딕셔너리로 저장합니다.
settings = {
    'run_id': RUN_ID, 'model_id': MODEL_ID, 'model_revision': REVISION,
    'prompt_version': PROMPT_VERSION, 'system_prompt': SYSTEM_PROMPT,
    'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS, 'dtype': 'float16',
    'gpu': torch.cuda.get_device_name(0), 'python': platform.python_version(),
    'torch': torch.__version__, 'transformers': transformers.__version__,
    'accelerate': accelerate.__version__, 'evaluation_set': 'rental-dev-3-v1',
    'rag': False, 'fine_tuning': False,
    'latency_definition': '입력 구성 · GPU 전송 · 생성 · 디코딩 포함. 파일 저장 · 화면 표시 · 모델 로딩 제외',
}

# 설정값들을 JSON 파일로 변환하여 생성해 둔 구글 드라이브 폴더에 저장합니다.
(SAVE_DIR/'settings.json').write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding='utf-8')

# 성공적으로 로드된 모델명과 사용 중인 GPU 장치명을 출력합니다.
print('모델:', MODEL_ID, '| GPU:', settings['gpu'])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

모델: Qwen/Qwen2.5-1.5B-Instruct | GPU: Tesla T4


## 4. 질문 → 답변 함수

매번 새로운 대화로 실행하므로 이전 질문 · 답변이 다음 질문에 섞이지 않음. 출력 토큰 한도에 걸린 답변은 `length_limit` 표시.


In [6]:
# 모델에게 질문을 던지고 답변과 소요 시간, 토큰 정보 등을 받아오는 함수를 정의합니다.
def ask(question):
    # GPU 작업 싱크를 맞추고 타이머를 시작합니다. (정확한 시간 측정을 위함)
    torch.cuda.synchronize()
    started = time.perf_counter()

    # 모델에 전달할 메시지 구조를 정의합니다. (시스템 프롬프트와 사용자의 질문 구성)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]

    # 대화 포맷을 모델이 이해할 수 있는 텍스트 형태(프롬프트)로 병합합니다.
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 텍스트 형태의 프롬프트를 숫자로 인코딩하여 GPU 메모리로 보냅니다.
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

    # gradient 계산을 하지 않아 메모리를 아끼는 추론 모드에서 모델에게 답변 생성을 지시합니다.
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,  # 매번 동일하고 가장 확률이 높은 답변을 내도록 설정
            pad_token_id=tokenizer.eos_token_id,
        )

    # 모델의 출력 결과 중 입력받은 프롬프트 부분을 제외하고 새로 생성된 답변 토큰만 잘라냅니다.
    generated = output[0, inputs['input_ids'].shape[1]:]

    # 숫자 토큰을 다시 사람이 읽을 수 있는 텍스트(한글)로 변환(디코딩)하고 앞뒤 공백을 제거합니다.
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # GPU 연산 종료를 대기하고 최종 걸린 시간을 측정합니다.
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    # 문장이 끝남을 알리는 특수 토큰(EOS)의 ID 목록을 가져옵니다.
    eos_ids = model.generation_config.eos_token_id
    eos_ids = eos_ids if isinstance(eos_ids, list) else [eos_ids]

    # 모델이 문장을 스스로 온전하게 끝마쳤는지, 아니면 최대 글자 수에 도달해서 끊겼는지 판단합니다.
    ended = len(generated) > 0 and int(generated[-1]) in eos_ids
    status = 'ok' if ended else 'length_limit'

    # 만약 답변이 아예 생성되지 않았다면 비어있다는 상태를 지정합니다.
    if not answer:
        status = 'empty_answer'

    # 분석 결과 데이터들을 하나의 파이썬 딕셔너리 형태로 반환합니다.
    return {
        'answer': answer,
        'latency_seconds': round(elapsed, 3), # 소요 시간 (소수점 셋째 자리까지)
        'input_tokens': int(inputs['input_ids'].shape[1]), # 질문의 토큰 수
        'output_tokens': int(len(generated)), # 답변의 토큰 수
        'status': status,
    }

## 5. 질문 3개 실행 · 즉시 저장

질문마다 결과를 JSONL 파일에 추가 저장. 이 셀을 다시 실행하면 새로운 파일을 만들어 중복 실험이 섞이지 않음.

첫 질문은 GPU 초기화 등의 영향으로 더 느릴 수 있으므로, 이 첫 실험에서 세 질문의 시간 차이를 성능 우열로 해석하지 않기.


In [7]:
# 결과를 한 줄씩 JSON 객체 형태로 저장할 JSONL 파일 경로를 정의합니다.
# 파일 이름의 중복을 피하기 위해 임의의 8자리 랜덤 코드를 포함합니다.
RESULT_FILE = SAVE_DIR / ('answers_' + uuid.uuid4().hex[:8] + '.jsonl')
results = []

print("결과 파일이 준비되었습니다:", RESULT_FILE)

결과 파일이 준비되었습니다: /content/drive/MyDrive/[01.공부]/[AI] KT Cloud TECH UP/[프로젝트 01]/05_result/인수/20261030_113056_b4e767/answers_2dca2b6b.jsonl


```markdown
### 2단계: 첫 번째 질문 (Q01) 실행 셀
첫 번째 질문을 모델에 던지고 결과를 확인한 뒤 바로 파일에 저장합니다.
```

In [8]:
# Q01 질문 정보를 설정합니다.
q_id, cat, q_text = 'Q01', '기본 설명', '전입신고와 확정일자는 무엇이 다른가요?'
row = {'run_id': RUN_ID, 'question_id': q_id, 'category': cat, 'question': q_text}

try:
    # 모델에 질문을 입력하고 답변 정보를 받아옵니다.
    row.update(ask(q_text))
except Exception as error:
    row.update({'answer': '', 'status': 'error', 'error': str(error)})

# 전체 결과 리스트에 담고 드라이브 파일에 실시간 기록합니다.
results.append(row)
with RESULT_FILE.open('a', encoding='utf-8') as f:
    f.write(json.dumps(row, ensure_ascii=False) + '\n')

# 결과를 출력합니다.
print('='*50)
print(f"[{q_id}] {q_text}")
print("답변:", row['answer'] or row.get('error'))
print(f"상태: {row['status']} | 소요 시간: {row.get('latency_seconds')}초")

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[Q01] 전입신고와 확정일자는 무엇이 다른가요?
답변: 전입 신고는 학교나 기관에서 학생이나 직원의 전입을 신청하는 과정입니다. 이 때, "확정일자"는 해당 신고가 정식으로 처리되고 확인된 날짜를 의미합니다.

즉, 전입 신고는 신청한 순간부터 시작되며, 확정일자는 신청 후 관리자가 이를 확인하고 처리한 날짜입니다. 따라서, 전입 신고는 신청일과 동시에 진행되지만, 확정일자는 관리자가 처리한 날짜가 됩니다.
상태: ok | 소요 시간: 5.591초


```markdown
### 3단계: 두 번째 질문 (Q04) 실행 셀
두 번째 질문을 실행하는 독립된 셀입니다.
```

In [9]:
# Q04 질문 정보를 설정합니다.
q_id, cat, q_text = 'Q04', '조건 판단', '이사는 했지만 전입신고는 하지 않았어요. 보증금 보호를 위해 무엇을 확인해야 하나요?'
row = {'run_id': RUN_ID, 'question_id': q_id, 'category': cat, 'question': q_text}

try:
    row.update(ask(q_text))
except Exception as error:
    row.update({'answer': '', 'status': 'error', 'error': str(error)})

results.append(row)
with RESULT_FILE.open('a', encoding='utf-8') as f:
    f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('='*50)
print(f"[{q_id}] {q_text}")
print("답변:", row['answer'] or row.get('error'))
print(f"상태: {row['status']} | 소요 시간: {row.get('latency_seconds')}초")

[Q04] 이사는 했지만 전입신고는 하지 않았어요. 보증금 보호를 위해 무엇을 확인해야 하나요?
답변: 보증금 보호를 위해서는 주소 변경 시에도 보증금을 반환받기 위한 조치를 취하는 것이 중요합니다. 이는 주소 변경 신청서와 함께 보증금을 반환받을 수 있도록 하는 것입니다. 또한, 이사 후에는 반드시 전입신고를 진행하여 주소 변경 사실을 공식적으로 인정받아야 합니다. 이는 보증금 반환을 받을 때도 중요한 절차입니다.
상태: ok | 소요 시간: 3.863초


```markdown
### 4단계: 세 번째 질문 (Q05) 실행 셀
마지막 질문을 실행하고 최종 저장을 마무리합니다.
```

In [10]:
# Q05 질문 정보를 설정합니다.
q_id, cat, q_text = 'Q05', '추가 질문', '저는 보증금을 우선해서 돌려받을 수 있나요?'
row = {'run_id': RUN_ID, 'question_id': q_id, 'category': cat, 'question': q_text}

try:
    row.update(ask(q_text))
except Exception as error:
    row.update({'answer': '', 'status': 'error', 'error': str(error)})

results.append(row)
with RESULT_FILE.open('a', encoding='utf-8') as f:
    f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('='*50)
print(f"[{q_id}] {q_text}")
print("답변:", row['answer'] or row.get('error'))
print(f"상태: {row['status']} | 소요 시간: {row.get('latency_seconds')}초")
print('\n' + '='*50)
print('모든 질문 실행 완료 및 파일 저장 완료:', RESULT_FILE)

[Q05] 저는 보증금을 우선해서 돌려받을 수 있나요?
답변: 네, 보증금은 일반적으로 사용자가 서비스를 이용할 때 지불해야 하는 예상 손실을 보장하기 위해 제공되는 금액입니다. 이 보증금은 사용자가 서비스를 종료하거나 불법적인 방법으로 서비스를 중단하면 반환될 수 있습니다. 하지만 실제로 보증금이 반환되지는 않을 수 있으며, 그 이유는 여러 가지가 있을 수 있습니다. 따라서 항상 사용자 정책이나 계약서를 확인하는 것이 중요합니다.
상태: ok | 소요 시간: 3.262초

모든 질문 실행 완료 및 파일 저장 완료: /content/drive/MyDrive/[01.공부]/[AI] KT Cloud TECH UP/[프로젝트 01]/05_result/인수/20261030_113056_b4e767/answers_2dca2b6b.jsonl


## 6. 수동 채점

1. 제공한 스프레드시트를 Drive에 업로드한 뒤 Google Sheets로 열기.
2. `채점표` 상단의 실험 ID에 위 RUN_ID 입력. 모델명 · 프롬프트 버전 확인.
3. 아래 `모델 답변 기록` 영역에 Q01 · Q04 · Q05 답변 · 응답 시간 · 상태 복사.
4. `기준 답변` 탭을 참고해 `채점표`의 15개 항목에 0 · 1 · 2 입력.
5. 치명적 오류 여부와 Q05의 실제 추가 질문 여부 입력. 문항별 점수와 총점은 자동 계산.

빈 답변 · 실행 오류 · 잘린 답변은 먼저 원인을 확인하고 완전한 답변을 다시 생성한 뒤 채점. 낮은 내용 점수는 그대로 기록.

**다음 실험:** 3번 셀의 SYSTEM_PROMPT와 PROMPT_VERSION만 변경한 뒤 3~5번 재실행. 모델 · 질문 · 생성 설정 유지. 채점표는 파일 사본을 만들어 별도 기록. 프롬프트에 기준 답변을 복사해 넣지 않기.

파일 저장 위치: 내 드라이브 → [01.공부] → [AI] KT Cloud TECH UP → [프로젝트 01] → result → 인수 → 실험 ID 폴더. `settings.json`에 설정, `answers_*.jsonl`에 답변 기록. RAM의 results만 보관하면 런타임 종료 시 사라지므로 Drive 파일 확인.
